In [2]:
import pandas as pd
import numpy as np

# --- Lire le fichier Excel (.xls) ---

df = pd.read_csv("/Users/serdarvarol/Desktop/PVL_L3-25_26/S2/sciDonne_L3/projet_vitamin_IA/bdds_pour_tests/paire2_vitamines_random_95.csv")

# --- Paramètre de la moyenne glissante ---
window = 10

# --- Colonnes numériques uniquement (on calcule mean/median/mode dessus) ---
num_cols = df.select_dtypes(include="number").columns.tolist()

# --- Fonctions de lissage ---
for col in num_cols:
    # Moyenne glissante
    df[f"{col}_ma{window}"] = df[col].rolling(window=window, min_periods=1).mean()
    # Médiane glissante
    df[f"{col}_median{window}"] = df[col].rolling(window=window, min_periods=1).median()
    # Mode glissant (valeur la plus fréquente sur la fenêtre)
    df[f"{col}_mode{window}"] = df[col].rolling(window=window, min_periods=1).apply(
        lambda x: x.mode().iloc[0] if len(x.mode()) else np.nan,
        raw=False
    )

from IPython.display import display
display(df)

# --- Sauvegarder le fichier final ---
output_path = "data2_with_ma_median_mode.csv"
df.to_csv(output_path, index=False)

output_path


,age,gender,bmi,smoking_status,alcohol_consumption,exercise_level,diet_type,sun_exposure,vitamin_a_percent_rda,vitamin_c_percent_rda,...,has_numbness_tingling_mode10,has_memory_problems_ma10,has_memory_problems_median10,has_memory_problems_mode10,has_pale_skin_ma10,has_pale_skin_median10,has_pale_skin_mode10,has_multiple_deficiencies_ma10,has_multiple_deficiencies_median10,has_multiple_deficiencies_mode10
0,30,Female,24.0,Former,Heavy,Sedentary,Pescatarian,Moderate,174.5,137.8,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,83,Male,18.2,Current,Moderate,Active,Omnivore,Moderate,161.7,131.1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,61,Female,22.1,Current,Heavy,Light,Omnivore,Moderate,209.2,146.2,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,51,Male,25.2,Never,NaN,Moderate,Omnivore,High,110.9,115.3,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,36,Male,23.3,Never,Moderate,Active,Pescatarian,Low,101.6,133.9,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
470,70,Male,31.2,Never,Heavy,Sedentary,Pescatarian,Moderate,43.6,10.0,...,0.0,0.5,0.5,0.0,0.2,0.0,0.0,0.7,1.0,1.0
471,56,Female,26.9,Never,Moderate,Moderate,Vegetarian,High,71.2,28.4,...,0.0,0.6,1.0,1.0,0.1,0.0,0.0,0.7,1.0,1.0
472,23,Male,19.8,Current,Moderate,Active,Pescatarian,High,64.6,27.5,...,0.0,0.6,1.0,1.0,0.2,0.0,0.0,0.8,1.0,1.0
473,77,Female,27.9,Current,Moderate,Active,Vegan,Moderate,81.7,18.3,...,1.0,0.6,1.0,1.0,0.2,0.0,0.0,0.8,1.0,1.0


'data2_with_ma_median_mode.csv'

In [3]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1) Lire ton fichier
file_path = "data2_with_ma_median_mode.csv"  # mets le bon chemin si besoin
df = pd.read_csv(file_path)

# 2) Features (comme sur ta capture)
features = [
    "vitamin_a_percent_rda",
    "hemoglobin_g_dl",
    "vitamin_c_percent_rda",
    "vitamin_e_percent_rda",
    "vitamin_b12_percent_rda",
    "vitamin_d_percent_rda",
    "folate_percent_rda",
    "calcium_percent_rda",
    "iron_percent_rda",
    "serum_vitamin_d_ng_ml",
    "serum_vitamin_b12_pg_ml",
    "serum_folate_ng_ml",
    "has_muscle_weakness",
    "has_night_blindness",
    "has_bleeding_gums",
    "has_fatigue",
    "has_bone_pain",
    "has_numbness_tingling",
    "has_memory_problems",
    "has_pale_skin",
    "diet_type"
]

target = "disease_diagnosis"

# 3) Vérifier que les colonnes existent
missing = [c for c in features + [target] if c not in df.columns]
if missing:
    raise ValueError(f"Colonnes manquantes dans ton CSV : {missing}")

# 4) X et y
X = df[features]
y = df[target]

# 5) Colonnes numériques / catégorielles
num_cols = X.select_dtypes(include="number").columns.tolist()
cat_cols = X.select_dtypes(exclude="number").columns.tolist()

# 6) Pré-traitement (simple)
# - Numériques : remplacer NaN par médiane
# - Catégorielles : remplacer NaN par valeur la plus fréquente + OneHot
preprocess = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), num_cols),
        ("cat", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_cols),
    ]
)

# 7) Modèle Random Forest (simple)
model = RandomForestClassifier(n_estimators=200, random_state=42)

# 8) Pipeline complet
clf = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", model)
])

# 9) Train / Test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.05, random_state=42, stratify=y
)

# 10) Entraîner + prédire
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

# 11) Résultats
print("Accuracy =", accuracy_score(y_test, y_pred))
print("\nRapport détaillé :\n", classification_report(y_test, y_pred))
print("\nMatrice de confusion :\n", confusion_matrix(y_test, y_pred))
print("version 1")


Accuracy = 0.9583333333333334

Rapport détaillé :
                       precision    recall  f1-score   support

              Anemia       1.00      1.00      1.00         5
             Healthy       1.00      0.75      0.86         4
     Night_Blindness       1.00      1.00      1.00         5
Rickets_Osteomalacia       0.83      1.00      0.91         5
              Scurvy       1.00      1.00      1.00         5

            accuracy                           0.96        24
           macro avg       0.97      0.95      0.95        24
        weighted avg       0.97      0.96      0.96        24


Matrice de confusion :
 [[5 0 0 0 0]
 [0 3 0 1 0]
 [0 0 5 0 0]
 [0 0 0 5 0]
 [0 0 0 0 5]]
version 1
